<a href="https://colab.research.google.com/github/Musab-908/ByteHost-A-Web-Hosting-Platform/blob/main/project_management_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Project Management Dashboard
**Step 1** → Run Cell 1 (installs libraries)  
**Step 2** → Run Cell 2 (upload CSV, renders full dashboard)

In [ ]:
!pip install plotly --quiet
print('Done')

Done


In [3]:
import io, warnings
import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files
from IPython.display import display, HTML
warnings.filterwarnings('ignore')


print('📂 Upload your project CSV...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
for col in ['start_date','due_date','completion_date']:
    if col in df.columns: df[col] = pd.to_datetime(df[col], errors='coerce')
for col in ['progress_pct','estimated_hours','actual_hours','budget_allocated','budget_spent']:
    if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
print(f'✅ Loaded {len(df)} tasks')


DARK   = '#0f1117'
PANEL  = '#1a1d27'
BORDER = '#2e3148'
T1     = '#e8eaf6'
T2     = '#9fa8da'
GREEN  = '#4caf50'; CYAN = '#26c6da'; BLUE = '#42a5f5'
ORANGE = '#ffa726'; RED  = '#ef5350'; PURPLE = '#ab47bc'; GREY = '#546e7a'
PHASES = [GREEN, CYAN, BLUE, ORANGE, PURPLE, RED, '#80cbc4', '#fff176']
SCLR   = {'complete':GREEN,'in progress':CYAN,'not started':GREY,'overdue':RED}


TODAY    = pd.Timestamp(datetime.today().date())
n        = len(df)
done     = (df['status'].str.lower()=='complete').sum()
wip      = (df['status'].str.lower()=='in progress').sum()
todo     = (df['status'].str.lower()=='not started').sum()
late     = ((df['due_date']<TODAY)&(df['status'].str.lower()!='complete')).sum() \
           if 'due_date' in df.columns else 0
prog     = df['progress_pct'].mean() if 'progress_pct' in df.columns else 0
tot_bud  = df['budget_allocated'].sum()
tot_spt  = df['budget_spent'].sum()
bud_pct  = (tot_spt/tot_bud*100) if tot_bud>0 else 0
bud_lbl  = 'Under Budget' if tot_spt<=tot_bud else 'Over Budget'
hvar     = df['actual_hours'].sum()-df['estimated_hours'].sum()


def card(lbl,val,sub='',c=BLUE):
    return (f'<div style="background:{PANEL};border:1px solid {BORDER};border-radius:10px;'
            f'padding:14px 16px;text-align:center;flex:1;min-width:110px">'
            f'<div style="font-size:24px;font-weight:700;color:{c}">{val}</div>'
            f'<div style="font-size:11px;color:{T2};margin-top:3px">{lbl}</div>'
            +(f'<div style="font-size:10px;color:{GREY};margin-top:1px">{sub}</div>' if sub else '')
            +'</div>')

display(HTML(
    f'<div style="background:{DARK};padding:18px;border-radius:12px;font-family:sans-serif;margin-bottom:6px">'
    f'<h2 style="color:{T1};margin:0 0 12px 0;font-size:19px">📊 Project Management Dashboard</h2>'
    f'<div style="display:flex;gap:8px;flex-wrap:wrap">'
    +card('Total Tasks',n,'',BLUE)
    +card('Complete',done,f'{done/n*100:.0f}% done',GREEN)
    +card('In Progress',wip,'',CYAN)
    +card('Not Started',todo,'',GREY)
    +card('Overdue',late,'',RED)
    +card('Progress',f'{prog:.1f}%','',PURPLE)
    +card('Budget Used',f'{bud_pct:.1f}%',bud_lbl,ORANGE if bud_pct>80 else GREEN)
    +card('Hours Variance',f'{hvar:+.0f}h','',RED if hvar>0 else GREEN)
    +'</div></div>'
))


def base(fig, h=370):
    fig.update_layout(
        paper_bgcolor=DARK, plot_bgcolor=PANEL,
        font=dict(color=T2,size=11), height=h,
        margin=dict(l=40,r=25,t=50,b=35),
        legend=dict(font=dict(color=T2,size=10),bgcolor='rgba(0,0,0,0)',bordercolor=BORDER),
    )
    fig.update_annotations(font_color=T1)
    return fig

def ax(fig, which=None, **kw):
    defaults = dict(showgrid=True,gridcolor=BORDER,zeroline=False,color=T2)
    defaults.update(kw)
    if which: fig.update_layout(**{which: defaults})
    return fig


fA = make_subplots(1,2,specs=[[{'type':'pie'},{'type':'xy'}]],
                   subplot_titles=['Task Status Breakdown','Avg Completion % by Phase'],
                   column_widths=[0.4,0.6])
sc = df['status'].str.title().value_counts()
fA.add_trace(go.Pie(
    labels=sc.index, values=sc.values, hole=0.55,
    marker_colors=[SCLR.get(s.lower(),GREY) for s in sc.index],
    textinfo='label+percent', textfont_size=11,
    hovertemplate='%{label}: %{value}<extra></extra>'
), 1,1)
if 'phase' in df.columns:
    ph = df.groupby('phase')['progress_pct'].mean().sort_values().reset_index()
    fA.add_trace(go.Bar(
        x=ph['progress_pct'], y=ph['phase'], orientation='h',
        marker_color=[PHASES[i%len(PHASES)] for i in range(len(ph))],
        text=[f'{v:.0f}%' for v in ph['progress_pct']], textposition='outside',
        showlegend=False, hovertemplate='%{y}: %{x:.0f}%<extra></extra>'
    ), 1,2)
base(fA); ax(fA,'xaxis',range=[0,120],ticksuffix='%'); ax(fA,'yaxis',showgrid=False)
fA.update_layout(showlegend=False); fA.show()


fB = make_subplots(1,2,
                   subplot_titles=['Budget — Allocated vs Spent by Phase','Tasks by Priority & Status'],
                   column_widths=[0.5,0.5])
if 'phase' in df.columns:
    bg = df.groupby('phase')[['budget_allocated','budget_spent']].sum()\
           .sort_values('budget_allocated').reset_index()
    fB.add_trace(go.Bar(name='Allocated', y=bg['phase'], x=bg['budget_allocated'],
                        orientation='h', marker_color=BLUE, opacity=0.85,
                        hovertemplate='Allocated: $%{x:,.0f}<extra></extra>'), 1,1)
    fB.add_trace(go.Bar(name='Spent', y=bg['phase'], x=bg['budget_spent'],
                        orientation='h', marker_color=GREEN, opacity=0.85,
                        hovertemplate='Spent: $%{x:,.0f}<extra></extra>'), 1,1)
if 'priority' in df.columns:
    ps = df.groupby(['priority','status']).size().reset_index(name='count')
    ps['priority'] = pd.Categorical(ps['priority'].str.title(),['High','Medium','Low'],ordered=True)
    ps = ps.sort_values('priority')
    seen = set()
    for _, row in ps.iterrows():
        st  = str(row['status'])
        stl = st.lower()
        fB.add_trace(go.Bar(
            name=st.title() if stl not in seen else None,
            x=[str(row['priority'])], y=[row['count']],
            marker_color=SCLR.get(stl,GREY),
            text=[row['count']], textposition='outside',
            showlegend=stl not in seen, legendgroup=stl,
            hovertemplate=f'{st.title()}: %{{y}}<extra></extra>'
        ), 1,2)
        seen.add(stl)
base(fB)
fB.update_layout(barmode='overlay')
ax(fB,'xaxis',tickprefix='$'); ax(fB,'yaxis',showgrid=False)
ax(fB,'xaxis2',showgrid=False); ax(fB,'yaxis2',title='Tasks')
fB.show()


fC = make_subplots(1,2,
                   subplot_titles=['Workload by Assignee','Estimated vs Actual Hours'],
                   column_widths=[0.5,0.5])
if 'assignee' in df.columns:
    wl = df.groupby('assignee').apply(lambda g: pd.Series({
        'Completed'  :(g['status'].str.lower()=='complete').sum(),
        'In Progress':(g['status'].str.lower()=='in progress').sum(),
        'Not Started':(g['status'].str.lower()=='not started').sum(),
        'Overdue'    :((g['due_date']<TODAY)&(g['status'].str.lower()!='complete')).sum()
                      if 'due_date' in g.columns else 0
    })).reset_index().sort_values('Completed')
    for col,clr in [('Completed',GREEN),('In Progress',CYAN),('Not Started',GREY),('Overdue',RED)]:
        fC.add_trace(go.Bar(name=col, y=wl['assignee'], x=wl[col], orientation='h',
                            marker_color=clr,
                            hovertemplate=f'{col}: %{{x}}<extra></extra>'), 1,1)
if {'estimated_hours','actual_hours','task_name'}.issubset(df.columns):
    hv = df[df['estimated_hours']>0].copy()
    hv['var'] = hv['actual_hours']-hv['estimated_hours']
    hv['bs']  = (hv['estimated_hours']/hv['estimated_hours'].max()*38+8).clip(8,46)
    fC.add_trace(go.Scatter(
        x=hv['estimated_hours'], y=hv['actual_hours'], mode='markers',
        marker=dict(size=hv['bs'], color=hv['var'].apply(lambda x: RED if x>0 else GREEN),
                    opacity=0.75, line=dict(width=1,color=BORDER)),
        text=hv['task_name'].str[:24],
        hovertemplate='<b>%{text}</b><br>Est: %{x}h / Act: %{y}h<extra></extra>',
        showlegend=False
    ), 1,2)
    mx = max(hv['estimated_hours'].max(), hv['actual_hours'].max())*1.1
    fC.add_trace(go.Scatter(x=[0,mx],y=[0,mx],mode='lines',name='Perfect estimate',
                            line=dict(color=ORANGE,dash='dash',width=1.5),hoverinfo='skip'), 1,2)
base(fC,h=390)
fC.update_layout(barmode='stack')
ax(fC,'xaxis',title='Tasks'); ax(fC,'yaxis',showgrid=False)
ax(fC,'xaxis2',title='Estimated Hours'); ax(fC,'yaxis2',title='Actual Hours')
fC.show()


if {'start_date','due_date','task_name','phase'}.issubset(df.columns):

    gdf = df.dropna(subset=['start_date','due_date']).copy()

    # ── Compress timeline: shift all dates so project starts at Day 0 ──────
    project_start = gdf['start_date'].min()
    gdf['day_start'] = (gdf['start_date'] - project_start).dt.days
    gdf['day_end']   = (gdf['due_date']   - project_start).dt.days
    gdf['duration']  = gdf['day_end'] - gdf['day_start']
    gdf['duration']  = gdf['duration'].clip(lower=1)   # minimum 1-day bar
    gdf['progress_days'] = (gdf['duration'] * gdf['progress_pct'] / 100).round(1)

    # ── Sort by phase order then start day ─────────────────────────────────
    phase_order = ['Initiation','Design','Development','Testing','Deployment','Closure']
    existing_phases = [p for p in phase_order if p in gdf['phase'].unique()]
    remaining = [p for p in gdf['phase'].unique() if p not in phase_order]
    all_phases = existing_phases + remaining
    gdf['phase_order'] = gdf['phase'].map({p:i for i,p in enumerate(all_phases)})
    gdf = gdf.sort_values(['phase_order','day_start'], ascending=[True, True])

    pmap = {p:PHASES[i%len(PHASES)] for i,p in enumerate(all_phases)}

    # ── Build Y labels: insert phase headers as separator rows ─────────────
    y_labels   = []   # final y-axis tick labels
    bar_data   = []   # (y_label, day_start, duration, progress_days, color, row)
    y_counter  = 0
    swimlane_shapes = []

    for phase in all_phases:
        tasks = gdf[gdf['phase']==phase]
        if tasks.empty: continue
        clr = pmap[phase]

        # Phase header row
        header_label = f'── {phase.upper()} ──'
        y_labels.append(header_label)
        swimlane_shapes.append(dict(
            type='rect', xref='x', yref='y',
            x0=0, x1=gdf['day_end'].max()+5,
            y0=y_counter-0.5, y1=y_counter+0.5,
            fillcolor=clr, opacity=0.08, line_width=0
        ))
        y_counter += 1

        for _, r in tasks.iterrows():
            label = r['task_name'][:32]
            y_labels.append(label)
            bar_data.append({
                'y'       : label,
                'x0'      : r['day_start'],
                'dur'     : r['duration'],
                'prog'    : r['progress_days'],
                'color'   : clr,
                'phase'   : phase,
                'status'  : r.get('status',''),
                'pct'     : r.get('progress_pct',0),
                'start_dt': r['start_date'].strftime('%b %d'),
                'end_dt'  : r['due_date'].strftime('%b %d'),
                'task'    : r['task_name'],
                'assignee': r.get('assignee',''),
            })
            y_counter += 1


    fG = go.Figure()


    for d in bar_data:
        fG.add_trace(go.Bar(
            x=[d['dur']], y=[d['y']], base=d['x0'],
            orientation='h', marker_color=d['color'],
            opacity=0.22, showlegend=False, hoverinfo='skip',
            marker_line_width=0
        ))


    legend_phases = set()
    for d in bar_data:
        show_leg = d['phase'] not in legend_phases
        legend_phases.add(d['phase'])
        st = str(d['status']).lower()
        bar_clr = SCLR.get(st, d['color'])
        fG.add_trace(go.Bar(
            x=[max(d['prog'], 0.5)],
            y=[d['y']], base=d['x0'],
            orientation='h',
            marker_color=bar_clr,
            opacity=0.90,
            name=d['phase'] if show_leg else None,
            legendgroup=d['phase'],
            showlegend=show_leg,
            hovertemplate=(
                f"<b>{d['task']}</b><br>"
                f"Assignee: {d['assignee']}<br>"
                f"Phase: {d['phase']}<br>"
                f"Status: {d['status']}<br>"
                f"Progress: {d['pct']:.0f}%<br>"
                f"Duration: Day {d['x0']} → {d['x0']+d['dur']} "
                f"({d['start_dt']} → {d['end_dt']})"
                f"<extra></extra>"
            )
        ))

    today_day = (TODAY - project_start).days
    if 0 <= today_day <= gdf['day_end'].max():
        fG.add_vline(x=today_day, line_color=ORANGE, line_dash='dash', line_width=1.5,
                     annotation_text='Today', annotation_font_color=ORANGE,
                     annotation_position='top')


    total_days = int(gdf['day_end'].max()) + 5
    tick_days, tick_labels = [], []
    for d in range(0, total_days, 14):
        dt = project_start + pd.Timedelta(days=d)
        tick_days.append(d)
        tick_labels.append(dt.strftime('%b %d'))

    n_tasks = len(bar_data)
    n_rows  = n_tasks + len(all_phases)
    row_h   = 22

    fG.update_layout(
        title=dict(text='📅 Project Gantt — Timeline (Days from Project Start)',
                   font=dict(color=T1,size=14), x=0.01),
        paper_bgcolor=DARK, plot_bgcolor=PANEL,
        font=dict(color=T2,size=10), barmode='overlay',
        height=max(520, n_rows*row_h + 120),
        margin=dict(l=250, r=120, t=60, b=50),
        xaxis=dict(
            showgrid=True, gridcolor=BORDER, zeroline=False, color=T2,
            tickvals=tick_days, ticktext=tick_labels,
            tickangle=-35, title='Calendar Date'
        ),
        yaxis=dict(
            showgrid=False, color=T2, autorange='reversed',
            tickfont=dict(size=9.5)
        ),
        legend=dict(
            title=dict(text='Phase', font=dict(color=T1,size=11)),
            font=dict(color=T2,size=10), bgcolor='rgba(0,0,0,0)',
            bordercolor=BORDER, x=1.01, y=1
        ),
        shapes=swimlane_shapes
    )
    fG.show()

print('\nDashboard complete!')

📂 Upload your project CSV...


Saving project_data (1).csv to project_data (1) (1).csv
✅ Loaded 35 tasks



Dashboard complete!
